# 02. 정규화 + 품질 — 원문을 표준 dict 로, 그리고 데이터 실태 확인

**무엇을 하나:** `00_raw` 원문을 `normalize_cookrcp` / `normalize_mafra` 로 **표준 레시피 dict**(이름·재료[{name,qty,unit}]·조리·영양·embed_text)로 바꾸고, 재료 분량·단위·영양 **채움률을 출력**한다. → `data/lab/02_norm.json`

**파서가 고치는 것:** 식약처 재료 텍스트(`RCP_PARTS_DTLS`)엔 요리명 줄, 섹션 머리말(`고명`, `●양념장 :`), 분량표기(`[1인분]`)가 섞여 있다. 옛 파서는 이를 **가짜 재료**로 만들거나 진짜 재료를 통째로 버려서 분량이 ~45%만 채워졌다. 고친 파서는 머리말을 걷어내고 `75g` 을 분량으로 잡아 **~95%** 로 올린다.

**영양 실태 (가짜로 안 채움):** 식약처는 영양 완비(99.9%), **농정원은 단백질/탄수/지방이 소스에 아예 없어 0%**. 이 0을 1이나 추정값으로 메우지 않고 **있는 그대로 드러낸다**(거짓 "단백질 미달"/거짓 통과 방지). 이게 검색 결과에 어떻게 보이는지는 05에서 확인.

**파이프라인 대응:** `s1_normalize.py`(파서) + `s2_clean.py`(품질 리포트).

In [ ]:
import sys,os,json
from pathlib import Path
from collections import defaultdict
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
try: sys.stdout.reconfigure(encoding='utf-8')
except Exception: pass
from dotenv import load_dotenv; load_dotenv()
from app.rag._normalize import normalize_cookrcp, normalize_mafra

raw=json.loads((B/'data'/'lab'/'00_raw.json').read_text(encoding='utf-8'))   # 정규화 전 원문
recs=[normalize_cookrcp(r,i) for i,r in enumerate(raw.get('cookrcp') or [])]
# 농정원: 기본/재료/과정을 RECIPE_ID 로 조인 (s1_normalize 와 동일)
m=raw.get('mafra') or {}
if m.get('basic'):
    ing=defaultdict(list); proc=defaultdict(list)
    for r in m.get('ingredient',[]): ing[str(r.get('RECIPE_ID'))].append(r)
    for r in m.get('process',[]):    proc[str(r.get('RECIPE_ID'))].append(r)
    for b in m['basic']:
        rid=str(b.get('RECIPE_ID')); recs.append(normalize_mafra(b, ing.get(rid,[]), proc.get(rid,[])))
(B/'data'/'lab'/'02_norm.json').write_text(json.dumps(recs,ensure_ascii=False),encoding='utf-8')

# 품질 실태 (s2_clean report 재현) — 소스별 분량/단위/영양 채움률
def pct(a,b): return f'{100*a//b}%' if b else '-'
by=defaultdict(list)
for r in recs: by[r['source']].append(r)
print('normalized:',len(recs),'-> data/lab/02_norm.json')
for src,rs in by.items():
    it=[i for r in rs for i in r['ingredients']]
    qty=sum(1 for i in it if i.get('qty') is not None); un=sum(1 for i in it if i.get('unit'))
    pcf=sum(1 for r in rs if any((r.get(k) or 0)>0 for k in ('protein','carb','fat')))
    print(f'  [{src}] {len(rs)}건 | 재료분량 {pct(qty,len(it))} 단위 {pct(un,len(it))} | 영양(P/C/F) {pct(pcf,len(rs))}')
s=recs[0]; print('embed_text ->', s['embed_text'][:160])

**출력 읽는 법**
- `재료분량 / 단위` = 그 소스 재료에 수치/단위가 붙은 비율. 식약처가 95% 가까우면 파서 수정이 반영된 것.
- `영양(P/C/F)` = 단백질·탄수·지방이 0이 아닌 레시피 비율. **농정원 0%는 데이터 수집 갭**(파서 문제 아님) — 농정원 영양결합 서비스를 추가 수집해야 채워진다.
- `embed_text` = 다음 노트북(03)에서 벡터가 될 문장. 재료 **이름**은 들어가지만 **분량(75g)은 안 들어간다**(메타로만).

> 전량(1,689건) 기준 정본 수치는 `data/pipeline/02_clean/report.json`. 이 노트북은 작은 표본이라 %가 조금 다를 수 있다.